# Day 6 — 평가 · 모니터링 대시보드

지금까지 Day1~5에서 우리는 "모델을 어떻게 불러오고, 어떻게 최적화하고, 어떻게 서빙할지"를 다뤘습니다. 하지만 MLOps의 핵심은 결국 **"지금 이 모델이 잘 동작하고 있는가?"를 숫자로 확인**할 수 있어야 한다는 것입니다.

Day6에서는 무거운 외부 API(GPT-4 judge 등)나 풀스택 모니터링 도구(Grafana/Prometheus) 없이, **로컬에서 바로 계산 가능한 지표들**만으로 최소한의 평가·모니터링 대시보드를 만들어봅니다.

다룰 내용:
1. **생성 품질 지표**: perplexity, 응답 길이, 키워드/포맷 체크
2. **추론 성능 모니터링**: TTFT/TPOT(Day2 복습), GPU 메모리, `tegrastats`로 GPU 사용률·온도 수집
3. **대시보드**: matplotlib으로 위 지표들을 한 화면에 모으기

> 왜 Grafana/Prometheus를 쓰지 않나요?
> 8GB 통합 메모리 보드에서 시계열 DB + 익스포터 + 대시보드 서버를 새로 띄우는 것은 이 랩의 스코프(로컬 1대 실습)에 비해 설치/리소스 부담이 큽니다. 지금 단계에서는 "무엇을 관측해야 하는가"를 이해하는 것이 먼저이고, 그 다음에 실제 운영 환경에서 풀스택 도구로 넘어가면 됩니다.

In [ ]:
import importlib.util, subprocess, sys

for pkg in ["matplotlib", "pandas"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"{pkg} 설치 중...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
    else:
        print(f"{pkg} 이미 설치됨")

import time
import math
import re
import threading
import subprocess
import shutil

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to(device)
model.eval()
print("모델 로드 완료")

## 1. 생성 품질 지표

외부 LLM judge 없이도 계산할 수 있는 지표 3가지를 씁니다.

- **Perplexity(PPL)**: 모델이 주어진 텍스트를 얼마나 "자연스럽다"고 여기는지의 척도. `loss`를 지수화(`exp`)해서 계산합니다. 낮을수록 모델이 해당 텍스트 분포에 익숙하다는 뜻입니다. 주의: 절대적인 "좋은 답변" 여부를 말해주진 않습니다(문법적으로 매끈하지만 내용이 틀린 답도 PPL은 낮을 수 있음). 어디까지나 유창성(fluency)의 대리 지표입니다.
- **응답 길이**: 토큰 수/글자 수. 너무 짧으면 성의 없는 답변, 너무 길면 장황하거나 반복(degeneration) 가능성을 의심할 신호가 됩니다.
- **키워드/포맷 체크**: 질문에 반드시 들어가야 할 핵심 단어가 응답에 포함되는지, 혹은 JSON 등 특정 포맷을 요구했다면 파싱 가능한지 확인하는 규칙 기반 체크입니다. 정교하지 않지만 회귀 테스트(regression test)처럼 "이전보다 나빠지지 않았는지"를 자동으로 잡아낼 수 있습니다.

In [ ]:
def compute_perplexity(model, tokenizer, text: str) -> float:
    """주어진 텍스트에 대한 모델의 perplexity를 계산합니다."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return math.exp(outputs.loss.item())


# 유창성 기준선을 잡기 위한 고정 평가 문장 (한국어 + 영어 섞어서)
eval_texts = [
    "인공지능 모델의 추론 성능을 모니터링하는 것은 실서비스 운영에서 매우 중요하다.",
    "The quick brown fox jumps over the lazy dog near the riverbank every morning.",
    "젯슨 오린 나노와 같은 엣지 디바이스에서는 메모리와 전력 제약을 고려한 모델 최적화가 필수적이다.",
    "asdf qwer zxcv 1234 !@#$ 의미없는 토큰 나열입니다",  # 일부러 '이상한' 문장도 넣어봄
]

print("=== 고정 평가 문장에 대한 Perplexity ===")
for t in eval_texts:
    ppl = compute_perplexity(model, tokenizer, t)
    print(f"PPL={ppl:8.2f} | {t[:45]}")

**관찰 포인트**: 마지막의 의미 없는 토큰 나열 문장이 다른 문장들보다 PPL이 훨씬 높게 나오는지 확인해보세요. 만약 그렇다면, perplexity가 최소한 "이 텍스트가 자연어답게 그럴듯한가"는 잘 잡아낸다는 뜻입니다. 반대로 문법은 맞지만 사실관계가 틀린 문장을 넣어보면 PPL이 낮게 나올 수도 있는데, 이게 바로 perplexity 단독으로는 '정답 여부'를 판단할 수 없는 이유입니다.

In [ ]:
def evaluate_response(response: str, tokenizer, expected_keywords=None, max_tokens_budget=None):
    """규칙 기반 응답 품질 체크: 길이, 키워드 포함 여부, 길이 예산 준수 여부"""
    n_tokens = len(tokenizer(response)["input_ids"])
    result = {
        "length_chars": len(response),
        "length_tokens": n_tokens,
    }
    if expected_keywords:
        result["keyword_hit"] = any(kw in response for kw in expected_keywords)
    if max_tokens_budget:
        result["within_length_budget"] = n_tokens <= max_tokens_budget
    return result

## 2. 추론 성능 모니터링

Day2에서 다뤘던 **TTFT(Time To First Token)**와 **TPOT(Time Per Output Token)**을 여기서 재사용합니다.

- **TTFT**: 요청을 보낸 시점부터 첫 토큰이 나올 때까지 걸린 시간. 사용자가 느끼는 "응답이 시작되는 체감 속도"를 결정합니다.
- **TPOT**: 첫 토큰 이후, 토큰 하나를 생성하는 데 평균적으로 걸리는 시간. 전체 응답을 다 받기까지의 체감 속도를 결정합니다.

여기에 더해 두 가지를 추가로 봅니다.

- **GPU 메모리 사용량**: `torch.cuda.memory_allocated()` / `memory_reserved()` — PyTorch가 실제로 할당한 텐서 메모리와, CUDA 캐싱 알로케이터가 예약해둔 메모리입니다. 8GB 통합 메모리 보드에서는 이 숫자가 OOM까지 얼마나 여유가 있는지 가늠하는 핵심 지표입니다.
- **GPU 사용률 / 온도**: PyTorch API로는 알 수 없고, Jetson 전용 도구인 `tegrastats`로 확인합니다.

In [ ]:
from transformers import TextIteratorStreamer


def measure_ttft_tpot(model, tokenizer, prompt: str, max_new_tokens: int = 120):
    """스트리밍 생성을 통해 TTFT/TPOT을 실측합니다 (Day2 방식 재사용)."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs, max_new_tokens=max_new_tokens, streamer=streamer, do_sample=False
    )

    start = time.perf_counter()
    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    first_token_time = None
    generated_text = ""
    for chunk in streamer:
        now = time.perf_counter()
        if first_token_time is None:
            first_token_time = now
        generated_text += chunk
    thread.join()
    end = time.perf_counter()

    n_tokens = len(tokenizer(generated_text)["input_ids"])
    ttft = (first_token_time - start) if first_token_time else None
    tpot = ((end - first_token_time) / max(n_tokens - 1, 1)) if first_token_time else None

    return {
        "prompt": prompt,
        "response": generated_text,
        "ttft_sec": ttft,
        "tpot_sec": tpot,
        "total_time_sec": end - start,
        "n_tokens": n_tokens,
    }


def gpu_memory_snapshot():
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_mb": torch.cuda.memory_allocated() / 1024**2,
        "reserved_mb": torch.cuda.memory_reserved() / 1024**2,
        "max_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
    }

In [ ]:
# 워밍업 (측정하지 않음) — 이 콜드스타트를 건너뛰지 않으면 첫 TTFT가 수십 초로 튈 수 있습니다.
# 실제로 이 노트북을 처음 검증할 때, 워밍업 없이 measure_ttft_tpot을 바로 호출했더니
# TTFT가 49초로 나온 적이 있습니다 (CC 8.7 PTX JIT 컴파일 비용 — Day1/Day2에서 본 것과 동일한 원인).
_ = measure_ttft_tpot(model, tokenizer, "안녕", max_new_tokens=5)
print("워밍업 완료 — 이후 측정값부터 신뢰할 수 있습니다.")

### `tegrastats` 출력 파싱하기

JetPack에는 GPU/CPU 사용률, 온도, 메모리 등을 실시간으로 보여주는 `tegrastats`라는 커맨드라인 도구가 기본 내장되어 있습니다 (`/usr/bin/tegrastats`). `jtop`과 달리 별도 서비스 설치나 `sudo` 권한 없이 바로 실행할 수 있습니다.

출력은 대략 이런 한 줄짜리 텍스트가 지정한 간격(ms)마다 stdout으로 계속 찍히는 형태입니다.

```
RAM 3266/7620MB (lfb 2x2MB) SWAP 512/3810MB (cached 12MB) CPU [15%@1510,12%@1510,10%@1510,8%@1510,5%@1510,3%@1510] EMC_FREQ 0% GR3D_FREQ 45% GR3DB_FREQ 0% VIC_FREQ 0% APE 174 cpu@45.5C soc2@44.7C soc0@44.5C tj@45.5C soc1@44.8C GPU@44.5C tboard@41C thermal@44.9C AO@45.5C
```

우리가 뽑아낼 필드:
- `RAM used/total MB` → 정규식 `RAM (\d+)/(\d+)MB`
- `GR3D_FREQ NN%` → GPU 코어 사용률(%). 정규식 `GR3D_FREQ (\d+)%`
- 온도 → `tj@` (junction 온도, SoC 전체 대표값)가 있으면 우선 사용하고 없으면 `GPU@`/`cpu@`로 폴백. 정규식 `(?:tj|GPU|gpu)@([\d.]+)C`

이 텍스트 스트림을 백그라운드 스레드에서 계속 읽어서 파싱하면, 파이썬 코드 안에서 생성(generate)이 진행되는 동안의 GPU 사용률/온도 변화를 시계열로 기록할 수 있습니다.

**주의(파일리스크 노트)**: `tegrastats`는 실행할 때 잠금 파일을 만들기 때문에, `Ctrl+C`나 `SIGTERM`으로만 죽이면 다음 실행 시 "이미 실행 중"이라며 충돌할 수 있습니다. 공식적으로는 `tegrastats --stop`으로 깨끗하게 종료하는 것이 권장되므로, 아래 코드에서는 `terminate()` 후 `tegrastats --stop`도 함께 호출합니다.

In [ ]:
class TegrastatsMonitor:
    """tegrastats 출력을 백그라운드에서 파싱해 GPU 사용률/온도/RAM을 시계열로 수집"""

    RAM_RE = re.compile(r"RAM (\d+)/(\d+)MB")
    GPU_RE = re.compile(r"GR3D_FREQ (\d+)%")
    TEMP_RE = re.compile(r"(?:tj|GPU|gpu)@([\d.]+)C")

    def __init__(self, interval_ms: int = 200):
        self.interval_ms = interval_ms
        self.proc = None
        self.thread = None
        self.records = []
        self._stop_flag = threading.Event()

    def _reader(self):
        for line in self.proc.stdout:
            if self._stop_flag.is_set():
                break
            ts = time.time()
            ram_m = self.RAM_RE.search(line)
            gpu_m = self.GPU_RE.search(line)
            temp_m = self.TEMP_RE.search(line)
            self.records.append({
                "time": ts,
                "ram_used_mb": int(ram_m.group(1)) if ram_m else None,
                "ram_total_mb": int(ram_m.group(2)) if ram_m else None,
                "gpu_util_pct": int(gpu_m.group(1)) if gpu_m else None,
                "temp_c": float(temp_m.group(1)) if temp_m else None,
            })

    def start(self):
        if shutil.which("tegrastats") is None:
            raise FileNotFoundError("tegrastats 실행 파일을 찾을 수 없습니다 (Jetson이 아닌 환경?)")
        self.records = []
        self._stop_flag.clear()
        self.proc = subprocess.Popen(
            ["tegrastats", "--interval", str(self.interval_ms)],
            stdout=subprocess.PIPE, text=True, bufsize=1,
        )
        self.thread = threading.Thread(target=self._reader, daemon=True)
        self.thread.start()

    def stop(self):
        self._stop_flag.set()
        if self.proc:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=2)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        # tegrastats 잠금 파일을 깨끗하게 정리 (없어도 에러 무시)
        subprocess.run(["tegrastats", "--stop"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if self.thread:
            self.thread.join(timeout=2)
        return self.records


# 짧게 동작 확인 (2초만 수집)
try:
    _monitor_test = TegrastatsMonitor(interval_ms=200)
    _monitor_test.start()
    time.sleep(2)
    _records = _monitor_test.stop()
    print(f"수집된 샘플 수: {len(_records)}")
    if _records:
        print("샘플 예시:", _records[0])
except FileNotFoundError as e:
    print(f"[경고] {e} — 이 노트북 이후 셀에서는 tegrastats 없이 진행합니다.")

## 3. 통합 벤치마크: 지표를 한 번에 수집하기

이제 여러 개의 테스트 프롬프트에 대해 (1) 생성 품질 지표, (2) TTFT/TPOT, (3) GPU 메모리를 한 번의 루프로 수집하고, 그 실행 시간 동안 `tegrastats`로 GPU 사용률/온도도 함께 기록합니다.

In [ ]:
# (프롬프트, 정답에 반드시 포함되길 기대하는 키워드) 쌍
test_prompts = [
    ("파이썬에서 리스트와 튜플의 차이를 한 문단으로 설명해줘.", ["리스트", "튜플"]),
    ("젯슨 오린 나노 같은 엣지 디바이스에서 LLM을 돌릴 때 주의할 점 3가지를 알려줘.", ["메모리", "젯슨"]),
    ("Explain what perplexity means in language models in two sentences.", ["perplexity"]),
    ("오늘 저녁 메뉴로 김치찌개를 추천하는 이유를 한 문장으로 말해줘.", ["김치찌개"]),
]

monitor = TegrastatsMonitor(interval_ms=200)
tegra_available = True
try:
    monitor.start()
except FileNotFoundError as e:
    tegra_available = False
    print(f"[경고] {e}")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

results = []
for prompt, keywords in test_prompts:
    perf = measure_ttft_tpot(model, tokenizer, prompt, max_new_tokens=120)
    quality = evaluate_response(perf["response"], tokenizer, expected_keywords=keywords, max_tokens_budget=150)
    mem = gpu_memory_snapshot()
    ppl = compute_perplexity(model, tokenizer, perf["response"])
    results.append({**perf, **quality, **mem, "perplexity": ppl})

tegra_records = monitor.stop() if tegra_available else []

df = pd.DataFrame(results)
df[["prompt", "ttft_sec", "tpot_sec", "length_tokens", "keyword_hit", "perplexity", "max_allocated_mb"]]

**관찰 포인트**
- `keyword_hit`이 `False`인 행이 있다면, 모델이 질문의 핵심을 놓쳤거나 답변이 너무 짧게 잘렸을 가능성을 의심해볼 수 있습니다.
- `ttft_sec`은 프롬프트 길이(입력 토큰 수)에 영향을 받습니다 — 프롬프트가 길수록 prefill 연산량이 늘어 TTFT가 커집니다.
- `perplexity`는 프롬프트마다 자연어/영어/설명형 답변 스타일에 따라 편차가 있을 수 있습니다. 절대값보다는 **같은 프롬프트 세트에 대해 시간이 지나며(혹은 모델 버전이 바뀌며) 어떻게 변하는지 추세**로 보는 것이 더 의미 있습니다.

## 4. 대시보드

지금까지 모은 지표를 하나의 그림으로 모아봅니다. 실서비스라면 이 값들이 시계열 DB에 쌓이고 Grafana 같은 도구가 시각화하겠지만, 이 랩의 스코프에서는 **matplotlib 한 장으로 충분**합니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
x = np.arange(len(df))

# 1) 응답별 Perplexity
axes[0, 0].bar(x, df["perplexity"], color="#4C72B0")
axes[0, 0].set_title("응답 Perplexity (낮을수록 유창)")
axes[0, 0].set_xlabel("프롬프트 idx")
axes[0, 0].set_ylabel("PPL")

# 2) 응답 길이 (토큰 수) + 키워드 히트 여부 색상 구분
colors = ["#55A868" if hit else "#C44E52" for hit in df["keyword_hit"]]
axes[0, 1].bar(x, df["length_tokens"], color=colors)
axes[0, 1].set_title("응답 길이 (초록=키워드 포함, 빨강=누락)")
axes[0, 1].set_xlabel("프롬프트 idx")
axes[0, 1].set_ylabel("토큰 수")

# 3) TTFT / TPOT
width = 0.35
axes[0, 2].bar(x - width / 2, df["ttft_sec"] * 1000, width, label="TTFT (ms)", color="#C44E52")
axes[0, 2].bar(x + width / 2, df["tpot_sec"] * 1000, width, label="TPOT (ms/token)", color="#8172B2")
axes[0, 2].set_title("추론 지연 시간")
axes[0, 2].set_xlabel("프롬프트 idx")
axes[0, 2].legend()

# 4) GPU 사용률 시계열 (tegrastats)
if tegra_records:
    t0 = tegra_records[0]["time"]
    times = [r["time"] - t0 for r in tegra_records]
    utils = [r["gpu_util_pct"] for r in tegra_records]
    axes[1, 0].plot(times, utils, color="#4C72B0")
    axes[1, 0].set_title("GPU 사용률 (%) - tegrastats")
    axes[1, 0].set_xlabel("경과 시간(s)")
    axes[1, 0].set_ylim(0, 100)
else:
    axes[1, 0].text(0.5, 0.5, "tegrastats 데이터 없음\n(Jetson 환경에서만 수집됨)", ha="center", va="center")
    axes[1, 0].set_title("GPU 사용률")

# 5) 온도 시계열 (tegrastats)
if tegra_records:
    temps = [r["temp_c"] for r in tegra_records]
    axes[1, 1].plot(times, temps, color="#C44E52")
    axes[1, 1].set_title("SoC/GPU 온도 (C) - tegrastats")
    axes[1, 1].set_xlabel("경과 시간(s)")
else:
    axes[1, 1].text(0.5, 0.5, "tegrastats 데이터 없음", ha="center", va="center")
    axes[1, 1].set_title("온도")

# 6) 프롬프트별 피크 GPU 메모리
axes[1, 2].bar(x, df["max_allocated_mb"], color="#8172B2")
axes[1, 2].set_title("누적 피크 GPU 메모리 (MB)")
axes[1, 2].set_xlabel("프롬프트 idx")
axes[1, 2].set_ylabel("MB")

plt.tight_layout()
plt.savefig("day6_dashboard.png", dpi=120)
plt.show()

print(f"8GB 중 관측된 피크 GPU 메모리 할당: {df['max_allocated_mb'].max():.1f} MB")

**관찰 포인트**
- `max_allocated_mb`는 매 프롬프트마다 계속 누적 측정되므로(`reset_peak_memory_stats`를 루프 밖에서 한 번만 호출), 대화가 길어질수록/배치가 커질수록 메모리가 어떻게 증가하는지 감을 잡을 수 있습니다. 참고로 이 값은 PyTorch가 잡고 있는 텐서 메모리만 보여주며, Jetson은 CPU/GPU가 **통합 메모리**를 쓰기 때문에 `tegrastats`의 RAM 사용량과 반드시 일치하지는 않습니다(OS, 다른 프로세스 몫도 포함되어 있기 때문). 실제 OOM 위험을 볼 때는 두 지표를 함께 참고하세요.
- GPU 사용률 그래프가 생성 구간 동안 높게 유지되다가 응답이 끝나면 뚝 떨어지는 패턴이 보이면 정상입니다. 만약 사용률이 낮은 채로 오래 걸린다면 CPU 바운드(토크나이징, 샘플링 로직 등) 구간이 병목일 수 있습니다.
- 온도가 실행 중 계속 상승 추세라면, 장시간 배치 추론 시 서멀 스로틀링(clock 저하)으로 TPOT이 점점 느려질 수 있다는 신호입니다 — 다음 단계로 발전시킨다면 이 스프린트처럼 몇 초짜리 벤치마크가 아니라 수 분 단위의 지속 부하 테스트로 확장해볼 수 있습니다.

## 정리

오늘 만든 것은 다음 세 층위로 요약됩니다.

| 층위 | 지표 | 계산 방법 |
|---|---|---|
| 품질 | perplexity, 응답 길이, 키워드 매칭 | `model(**inputs, labels=...)`, 토크나이저, 문자열 규칙 |
| 성능 | TTFT, TPOT, GPU 메모리 | `TextIteratorStreamer` + 타이밍, `torch.cuda.memory_*` |
| 시스템 | GPU 사용률, 온도 | `tegrastats` subprocess 파싱 |

**한계와 다음 단계**
- 여기서 쓴 perplexity/키워드 체크는 실제 서비스 품질 평가로 쓰기엔 거칠고(coarse), 사람 평가나 더 정교한 LLM-judge를 대체하지 못합니다. 다만 회귀 여부를 자동으로 감지하는 "스모크 테스트" 용도로는 충분합니다.
- 지금은 노트북을 실행할 때만 지표가 수집되는 1회성 스냅샷입니다. 이를 지속적으로 쌓으려면 `df`를 CSV/Parquet으로 append하거나, 가벼운 SQLite에 기록하는 정도로도 "미니 모니터링 파이프라인"을 만들 수 있습니다 — 굳이 처음부터 Prometheus/Grafana를 설치할 필요는 없습니다.
- `tegrastats` 파싱은 JetPack 버전에 따라 필드 이름이 조금씩 다를 수 있습니다(`tj@` 대신 다른 zone 이름일 수도 있음). 실제 보드에서 `tegrastats` 원본 출력을 한 번 눈으로 확인하고 정규식을 맞춰보는 것을 권장합니다.